In [82]:
import base64
import requests
import os
import glob
import json
import time
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
import os.path
from selenium import webdriver
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from typing import List, Optional
import sys
# Define the path to the utils folder
utils_folder = '../utils/'

# Add the utils folder to the system path
sys.path.append(os.path.abspath(utils_folder))
import openai 

In [83]:
# Load environment variables
load_dotenv()

True

In [84]:
# Initialize OpenAI client with API key
client = OpenAI()
OpenAI.api_key = os.getenv('OPENAI_API_KEY')

In [85]:
class MonetizationData(BaseModel):
    MonetizationMethod: list[str]
    DonationLinks: list[str]
    DonationMethods: list[str]
    SuggestedDonationAmounts: list[str]
    RecurringDonationsOption: str
    TextRelatedToDonations: str
    ButtonsOrWidgets: str
    PlacementOfDonationSection:str
    ThirdPartyPlatforms: list[str]
    CallToActionMessages: list[str]
    PaymentPlatformIntegration:list[str]
    PaymentButtonWidgetLocation:str
    AcceptedPaymentMethods: list[str]
    PricingModels: list[str]
    CurrencySupport: list[str]
    SubscriptionPlans: list[str]   

class MonetizationDataList(BaseModel):
    MonetizationDataList: list['MonetizationData']


class ProductData(BaseModel):
    ProductName: str
    ProductDescription: Optional[str]
    Price: Optional[str]
    DiscountOrOffers: Optional[str]
    ProductCategory: str
    Subcategory: Optional[str]
    ProductPageURL: str
    RelatedProducts: List[str]
    BrandName: str
    SellerInformation: Optional[str]
    ShippingInformation: Optional[str]
    ProductImageURLs: List[str]
    VideoURLs: Optional[List[str]]
    PromotionalBannersOrAds: Optional[str]
    SpecialCampaignsOrDiscounts: Optional[str]
    ProductTagsOrKeywords: Optional[List[str]]
    WebpageTitle: str
    MetaDescription: Optional[str]
    LastUpdatedDate: str
    PublisherOrWebsiteOwner: str

class ProductDataList(BaseModel):
    ProductDataList: List[ProductData]

In [86]:
def generate_prompt(page_text, links, prompt_type):
    """
    Generates a structured prompt based on the type of information to extract 
    from a webpage, such as donations, product sales, or advertising details.

    Args:
        page_text (str): The text content of the webpage.
        links (str): Links found on the webpage.
        prompt_type (str): Type of prompt to generate ("donations", "product_sales", or "advertisings").

    Returns:
        str: A formatted prompt string for extracting specific data from the webpage.
    """
    # Generate prompt for donation-related information extraction
    if prompt_type == "donations":
        prompt = f"""
        You are an expert in identifying payment and donation-related data from websites. 
        The website may be in English or another language. Your task is to extract the 
        following monetization-related information, translate all text into English, and 
        return it in the format provided below:

        - **MonetizationMethod:** [Type of monetization method used (e.g., donations, subscriptions, pay-per-article, ads, etc.)]
        - **DonationLinks:** [URLs directing users to donation pages or forms]
        - **DonationMethods:** [Types of donation methods accepted (e.g., PayPal, credit cards, cryptocurrency)]
        - **SuggestedDonationAmounts:** [Any preset or suggested donation amounts]
        - **RecurringDonationsOption:** [Information on one-time vs recurring donation options]
        - **TextRelatedToDonations:** [Specific phrases or text promoting donations (e.g., 'Support us', 'Donate now'), translated into English]
        - **ButtonsOrWidgets:** [Visual elements facilitating donations (e.g., donation buttons, widgets)]
        - **PlacementOfDonationSection:** [Location of donation-related elements on the page (e.g., header, footer, sidebar)]
        - **ThirdPartyPlatforms:** [Integrations with third-party platforms (e.g., Patreon, Ko-fi, GoFundMe)]
        - **CallToActionMessages:** [Prominent messages encouraging donations, translated into English]
        - **PaymentPlatformIntegration:** [Payment platforms integrated on the website (e.g., Stripe, PayPal, Square)]
        - **PaymentButtonWidgetLocation:** [Location of payment buttons or widgets (e.g., top, bottom, sidebar)]
        - **AcceptedPaymentMethods:** [Methods accepted for payments (e.g., credit cards, bank transfers, cryptocurrency)]
        - **PricingModels:** [Pricing models used (e.g., pay-per-article, subscriptions, freemium models)]
        - **CurrencySupport:** [Currencies supported for payments or donations]
        - **SubscriptionPlans:** [Details about subscription plans (e.g., pricing tiers, duration, benefits), translated into English]

        Below is the text scraped from the webpage:

        **Webpage text:**
        {page_text}

        Additionally, here are the links found on the page. Please extract and validate any links related to payments or donations:
        {links}
        """
    # Generate prompt for product sales-related information extraction
    elif prompt_type == "product_sales":
        prompt = f"""
        You are given the text of a webpage that lists products sold online. 
        Your task is to extract specific data from the webpage, structured as follows:
           - Product Name
           - Product Description (if available)
           - Price (if available)
           - Discount or Offers (if available)
           - Product Category
           - Subcategory (if available)
           - Product Page URL
           - Related Products or Links to Other Products
           - Brand Name
           - Seller Information (if applicable)
           - Shipping Information (if mentioned)
           - Product Image URL(s)
           - Video URL(s) (if present)
           - Promotional Banners or Ads (if present)
           - Special Campaigns or Discounts
           - Product Tags or Keywords (if listed)
           - Webpage Title
           - Meta Description (if available)
           - Last Updated Date
           - Publisher or Website Owner

        The webpage text is:
        {page_text}

        Additionally, here are the links found on the page:
        {links}

        Please extract the information and present it in a structured format.
        """
        
    
    return prompt


In [87]:
def generate_completions(prompt, prompt_type, model='gpt-4o-mino', max_tokens=20000):
    """
    Sends a request to the OpenAI API to generate news article completions based on the given prompt.
    
    Args:
        prompt (str): A text prompt to be used by the model.
        prompt_type (str): The type of prompt (e.g., "donations", "product_sales", "advertisings").
        model (str): The model name to be used. Defaults to 'gpt-4o-mino'.
        max_tokens (int): The maximum number of tokens to generate. Defaults to 20000.

    Returns:
        dict: A parsed response containing the generated news article.
    """ 
    # Define the response structure based on prompt type
    if prompt_type == "donations":
        response_structure = MonetizationDataList
    elif prompt_type == "product_sales":
        response_structure = ProductDataList
    elif prompt_type == "advertisings":
        response_structure = AdvertisingDataList
    else:
        raise ValueError("Invalid prompt_type. Must be one of: 'donations', 'product_sales', 'advertisings'.")

    # Sending request to the OpenAI API
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [{"type": "text", "text": prompt}]
            }
        ],
        response_format=response_structure,
        max_tokens=max_tokens,
        temperature=0
    )
    
    # Return the parsed message containing the generated news article
    return completion.choices[0].message.parsed

In [88]:
def create_dataframe(df, prompt_type, monetization_url, source_url):
    """
    Extract monetization data from websites and store it in a DataFrame.

    Args:
        df (pd.DataFrame): DataFrame containing 'Monetization_url' and 'Original_url' columns.
        prompt_type (str): Type of prompt to be generated, indicating the kind of information to be extracted
                           (e.g., 'donations', 'product_sales', 'advertisings').
        monetization_url (str): Column name for URLs pointing to monetization-related pages.
        source_url (str): Column name for source URLs from which data is being extracted.

    Returns:
        pd.DataFrame: DataFrame with extracted monetization information.
    """
    data = []

    for index, row in df.iterrows():
        try:
            # Initialize WebDriver to fetch webpage content with JavaScript rendering
            driver = webdriver.Chrome()
            driver.get(row[monetization_url])  # Load the page

            # Get the page source after JavaScript rendering
            html = driver.page_source
            driver.quit()

            # Parse HTML content
            soup = BeautifulSoup(html, "html.parser")

            # Extract all text and links from the webpage
            page_text = soup.get_text()  # Extracts all text from the webpage
            links = [a['href'] for a in soup.find_all('a', href=True)]  # Extracts all hyperlinks

            # Generate prompt and get completions from the model
            prompt = generate_prompt(page_text, links, prompt_type)
            results = generate_completions(prompt, prompt_type, model='gpt-4o-mini', max_tokens=15000)

            if prompt_type == "donations":
                # Process each result from the model output for donations
                for monetization_data_item in results.MonetizationDataList:
                    donation_dict = {
                        'Source_Url': row[source_url],
                        'Monetization_Url': row[monetization_url],
                        'Monetization_Method': monetization_data_item.MonetizationMethod,
                        'Donation_Links': monetization_data_item.DonationLinks,
                        'Donation_Methods': monetization_data_item.DonationMethods,
                        'Suggested_Donation_Amounts': monetization_data_item.SuggestedDonationAmounts,
                        'Recurring_Donations_Option': monetization_data_item.RecurringDonationsOption,
                        'Text_Related_To_Donations': monetization_data_item.TextRelatedToDonations,
                        'Buttons_Or_Widgets': monetization_data_item.ButtonsOrWidgets,
                        'Placement_Of_Donation_Section': monetization_data_item.PlacementOfDonationSection,
                        'ThirdParty_Platforms': monetization_data_item.ThirdPartyPlatforms,
                        'Call_To_Action_Messages': monetization_data_item.CallToActionMessages,
                        'PaymentPlatform_Integration': monetization_data_item.PaymentPlatformIntegration,
                        'Payment_Button_Widget_Location': monetization_data_item.PaymentButtonWidgetLocation,
                        'Accepted_Payment_Methods': monetization_data_item.PricingModels,
                        'Pricing_Models': monetization_data_item.CurrencySupport,
                        'Currency_Support': monetization_data_item.SubscriptionPlans,
                        'Subscription_Plans': monetization_data_item.SubscriptionPlans
                    }
                    data.append(donation_dict)
            
            elif prompt_type == "product_sales":
                # Process each result from the model output for product sales
                for product_data_item in results.ProductDataList:
                    product_dict = {
                        'Source_Url': row[source_url],
                        'Monetization_Url': row[monetization_url],
                        'Product_Name': product_data_item.ProductName,
                        'Product_Description': product_data_item.ProductDescription,
                        'Price': product_data_item.Price,
                        'Discount_Or_Offers': product_data_item.DiscountOrOffers,
                        'Product_Category': product_data_item.ProductCategory,
                        'Subcategory': product_data_item.Subcategory,
                        'Product_Page_URL': product_data_item.ProductPageURL,
                        'Related_Products': product_data_item.RelatedProducts,
                        'Brand_Name': product_data_item.BrandName,
                        'Seller_Information': product_data_item.SellerInformation,
                        'Shipping_Information': product_data_item.ShippingInformation,
                        'Product_Image_URLs': product_data_item.ProductImageURLs,
                        'Video_URLs': product_data_item.VideoURLs,
                        'Promotional_Banners_Or_Ads': product_data_item.PromotionalBannersOrAds,
                        'Special_Campaigns_Or_Discounts': product_data_item.SpecialCampaignsOrDiscounts,
                        'Product_Tags_Or_Keywords': product_data_item.ProductTagsOrKeywords,
                        'Webpage_Title': product_data_item.WebpageTitle,
                        'Meta_Description': product_data_item.MetaDescription,
                        'Last_Updated_Date': product_data_item.LastUpdatedDate,
                        'Publisher_Or_Website_Owner': product_data_item.PublisherOrWebsiteOwner
                    }
                    data.append(product_dict)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching {row[monetization_url]}: {e}")
            continue
        except Exception as e:
            print(f"Error processing data for {row[monetization_url]}: {e}")
            continue

    # Return the DataFrame with collected monetization data
    return pd.DataFrame(data)

In [89]:
# Specify the file path
filepath = "../data/monetisation_urls_datasets.xlsx"

# Load the Excel file into a DataFrame
df = pd.read_excel(filepath)

# Display the first few rows to verify
print(df.head())


                                        Original_Url  \
0  'https://vivementle9juin.fr/projet/europe-qui-...   
1   'https://pravda-gr.com/world/2024/05/09/6152....   
2   'https://triglavmedia.si/novice/okolje-ekolog...   
3   'https://euronewsbulgaria.com/news/27068/kiev...   
4   'https://www.dz-rs.si/wps/portal/Home/seje/ev...   

                                  Monetization_Url  \
0      https://adhesions.rassemblementnational.fr/   
1                                              NaN   
2  https://triglavmedia.si/podprite-svobodo-govora   
3                                              NaN   
4                                              NaN   

               Type_Of_Revenue Patner_Names  
0                   Membership          NaN  
1                          NaN          NaN  
2  Direct Donation, Membership          NaN  
3                          NaN          NaN  
4                          NaN          NaN  


In [92]:
# Drop rows with missing values in the 'Monetization_Url' column
df = df.dropna(subset=['Monetization_Url'])

# Define the revenue types to exclude
exclude_types = ['advertising', 'partner funded', 'product sale', 'magazine sale']

# Filter out rows where 'Type_Of_Revenue' matches any of the excluded types (case insensitive)
df1 = df[~df['Type_Of_Revenue'].astype(str).str.lower().isin(exclude_types)]

# Verify the filtered DataFrame
df1.shape

(101, 4)

In [10]:
import ast

# Generate the initial DataFrame with donation-related data
final_df = create_dataframe(df1, "donations", 'Monetization_Url', 'Original_Url')

# Convert string representations of lists in 'Donation_Links' to actual list objects
final_df['Donation_Links'] = final_df['Donation_Links'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Filter rows where 'Donation_Links' contains at least one link, then explode the list into separate rows
df_exploded = final_df[final_df['Donation_Links'].apply(lambda x: isinstance(x, list) and len(x) > 0)].explode('Donation_Links')

# Run create_dataframe again on the exploded DataFrame, using 'Donation_Links' as the monetization URL
results_df = create_dataframe(df_exploded, "donations", 'Donation_Links', 'Source_Url')

# Display the results DataFrame
print(results_df.head())

Error processing data for /selfhelp/home: Message: invalid argument
  (Session info: chrome=131.0.6778.85)
Stacktrace:
	GetHandleVerifier [0x00007FF7D6786CB5+28821]
	(No symbol) [0x00007FF7D66F3840]
	(No symbol) [0x00007FF7D65955B9]
	(No symbol) [0x00007FF7D6583051]
	(No symbol) [0x00007FF7D65812FD]
	(No symbol) [0x00007FF7D6581B3C]
	(No symbol) [0x00007FF7D659885A]
	(No symbol) [0x00007FF7D66301FE]
	(No symbol) [0x00007FF7D660F2FA]
	(No symbol) [0x00007FF7D662F412]
	(No symbol) [0x00007FF7D660F0A3]
	(No symbol) [0x00007FF7D65DA778]
	(No symbol) [0x00007FF7D65DB8E1]
	GetHandleVerifier [0x00007FF7D6ABFCAD+3408013]
	GetHandleVerifier [0x00007FF7D6AD741F+3504127]
	GetHandleVerifier [0x00007FF7D6ACB5FD+3455453]
	GetHandleVerifier [0x00007FF7D684BDBB+835995]
	(No symbol) [0x00007FF7D66FEB5F]
	(No symbol) [0x00007FF7D66FA814]
	(No symbol) [0x00007FF7D66FA9AD]
	(No symbol) [0x00007FF7D66EA199]
	BaseThreadInitThunk [0x00007FFCAB2B259D+29]
	RtlUserThreadStart [0x00007FFCABE2AF38+40]

Error proc

In [11]:
# Concatenate the two DataFrames
df_concat = pd.concat([final_df, results_df], ignore_index=True)

# Save the combined DataFrame to a CSV file
output_path = "../data/monetization_dataset_final.csv"
df_concat.to_csv(output_path, index=False)

# Print confirmation message
print(f"Data successfully saved to {output_path}")

Data successfully saved to ../data/monetization_dataset_final.csv


In [12]:
# List of revenue types to filter from the main DataFrame
key_list = ['Magzine Sale', 'Advertising', 'Patner Funded', 'Product Sale', 'Patner funded']

# Filter df to include only rows with 'Type_Of_Revenue' values in key_list
df2 = df[df['Type_Of_Revenue'].isin(key_list)]

# Separate DataFrame for 'Product Sale' and 'Magzine Sale' types
product_sales_df = df2[df2['Type_Of_Revenue'].isin(['Product Sale', 'Magzine Sale'])]

# Separate DataFrame for 'Advertising' type
advertising_df = df2[df2['Type_Of_Revenue'] == 'Advertising']

# Print confirmation to check the number of rows in each filtered DataFrame
print(f"Product Sales DataFrame rows: {len(product_sales_df)}")
print(f"Advertising DataFrame rows: {len(advertising_df)}")

Product Sales DataFrame rows: 6
Advertising DataFrame rows: 39


In [13]:
# Extract data from product_sales_df using the 'product_sales' prompt type
product_sales_extracted_data_df = create_dataframe(
    product_sales_df, "product_sales", 'Monetization_Url', 'Original_Url'
)

# Save the extracted data to a CSV file
product_sales_extracted_data_df.to_csv("../data/product_sales_extracted_data_final.csv", index=False)

print("Product sales data extraction completed and saved to product_sales_extracted_data.csv")

Product sales data extraction completed and saved to product_sales_extracted_data.csv


In [ ]:
product_sales_extracted_data_df.to_csv("../data/product_sales_extracted_data.csv", index=False)
Advertising_df.to_csv("../data/advertising_extracted_data.csv", index=False)

In [17]:
type(results_df)

pandas.core.frame.DataFrame

In [25]:
# Mark all duplicates as True
duplicates = df_concat['Monetization_Url'].duplicated(keep=False)
df[duplicates]

C:\Users\Neeraj Kulkarni\AppData\Local\Temp\ipykernel_28136\1117725012.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[duplicates]


,Original_Url,Monetization_Url,Type_Of_Revenue,Patner_Names
11,"'https://archive.ph/NDvuk',",https://buymeacoffee.com/archive.today,Direct Donation,NaN
17,'https://sensa.metropolitan.si/zdrave-odlocit...,https://www.adriamedia.si/naroci-revijo/,Magzine Sale,NaN
18,'https://sensa.metropolitan.si/zdrave-odlocit...,https://www.adriamedia.si/oglasevanje/,Advertising,NaN
20,'https://www.izraelisot.com/2024/02/20/fyejti...,https://www.izraelisot.com/membership-options/,Membership,NaN
21,'https://trud.bg/%D1%85%D1%8A%D1%80%D0%B2%D0%...,https://trud.bg/%D1%80%D0%B5%D0%BA%D0%BB%D0%B0...,Advertising,NaN
...,...,...,...,...
237,'https://observador.pt/liveblogs/caso-gemeas-...,https://observador.pt/assinar/premium/?utm_sou...,Subcription,NaN
241,"'https://archive.ph/lOTRI',",https://buymeacoffee.com/archive.today,Direct Donation,NaN
243,'https://sso.globalnoticias.pt/realms/GMG_PRD...,https://www.dn.pt/assinaturas/,Subcription,NaN
246,"'https://noa.al/lajmi/2024/05/2434755.html',",https://noa.al/faqe/marketing&pr.html,Advertising,NaN


(315, 18)

In [72]:
mon_df = pd.read_csv("../data/monetization_dataset_final.csv")

In [73]:
mon_df.shape

(315, 18)

In [74]:
def clean_string_of_list(column):
    def parse_and_convert(value):
        try:
            # Safely evaluate the string as a Python literal
            parsed = ast.literal_eval(value)
            # Check if it's a list, then join as comma-separated values
            if isinstance(parsed, list):
                return ", ".join(map(str, parsed))
        except (ValueError, SyntaxError):
            # Return original value if parsing fails
            return value
    return column.apply(parse_and_convert)

In [75]:
import ast

In [76]:
mon_df['Monetization_Method'] = clean_string_of_list(mon_df['Monetization_Method'])
mon_df['Donation_Links'] = clean_string_of_list(mon_df['Donation_Links'])
mon_df['Donation_Methods'] = clean_string_of_list(mon_df['Donation_Methods'])
mon_df['ThirdParty_Platforms'] = clean_string_of_list(mon_df['ThirdParty_Platforms'])
mon_df['Call_To_Action_Messages'] = clean_string_of_list(mon_df['Call_To_Action_Messages'])
mon_df['PaymentPlatform_Integration'] = clean_string_of_list(mon_df['PaymentPlatform_Integration'])
mon_df['Accepted_Payment_Methods'] = clean_string_of_list(mon_df['Accepted_Payment_Methods'])
mon_df['Pricing_Models'] = clean_string_of_list(mon_df['Pricing_Models'])
mon_df['Currency_Support'] = clean_string_of_list(mon_df['Currency_Support'])
mon_df['Subscription_Plans'] = clean_string_of_list(mon_df['Subscription_Plans'])
mon_df['Suggested_Donation_Amounts'] = clean_string_of_list(mon_df['Suggested_Donation_Amounts'])

In [77]:
mon_df.duplicated(keep=False).sum()

np.int64(109)

In [78]:
dd =  mon_df.drop_duplicates()

In [56]:
dd.to_csv("ty_without_duplicates.csv",index=False,encoding = 'utf-8')

In [79]:
len(dd)

252

In [80]:
dd1 = dd.drop_duplicates(subset= ['Source_Url','Monetization_Url','Placement_Of_Donation_Section'],keep='first')

In [61]:
dd2 = dd1.drop(columns = ['Accepted_Payment_Methods','ThirdParty_Platforms','PaymentPlatform_Integration'])

In [66]:
dd2 =pd.read_csv("ty2_no_dup.csv",encoding = 'utf-8')

In [81]:
dd1

,Source_Url,Monetization_Url,Monetization_Method,Donation_Links,Donation_Methods,Suggested_Donation_Amounts,Recurring_Donations_Option,Text_Related_To_Donations,Buttons_Or_Widgets,Placement_Of_Donation_Section,ThirdParty_Platforms,Call_To_Action_Messages,PaymentPlatform_Integration,Payment_Button_Widget_Location,Accepted_Payment_Methods,Pricing_Models,Currency_Support,Subscription_Plans
0,'https://vivementle9juin.fr/projet/europe-qui-...,https://adhesions.rassemblementnational.fr/,"donations, subscriptions",https://adhesions.rassemblementnational.fr/don...,"PayPal, credit cards","250 €, 100 €, 50 €, 30 €, 10 €",Donations can be one-time or recurring based o...,Support the RN!,Download the PDF form for donations and member...,Sidebar and footer of the page.,PayPal,"Join us!, Donate now!, Support us!",PayPal,Top of the donation section.,subscriptions,€,Prestige Membership: 250 € (85 € after tax red...,Prestige Membership: 250 € (85 € after tax red...
1,'https://triglavmedia.si/novice/okolje-ekolog...,https://triglavmedia.si/podprite-svobodo-govora,"donations, subscriptions","https://ko-fi.com/triglavmedia, https://trigla...","PayPal, credit cards, cryptocurrency","1€/month, 12€/year",Monthly or yearly support membership available.,Support freedom of speech with a donation or m...,Donate button on the Ko-fi platform.,Footer and dedicated donation page.,Ko-fi,"Support us, Donate now","PayPal, Ko-fi",Bottom of the page and on the donation page.,subscriptions,"EUR, USD",1€/month or 12€/year for membership,1€/month or 12€/year for membership
2,'https://lafranceinsoumise.fr/europeennes-202...,https://actionpopulaire.fr/contributions/,"donations, subscriptions",,credit cards,,Recurring donations are available.,Become a supporter of La France insoumise — Po...,Donation button available on the page.,Footer of the page.,,"Support us, Donate now",Stripe,Bottom of the page.,subscriptions,EUR,"Monthly subscription for €5, Annual subscripti...","Monthly subscription for €5, Annual subscripti..."
3,'https://lafranceinsoumise.fr/europeennes-202...,https://actionpopulaire.fr/dons/,donations,,PayPal,,One-time donations only,Make a donation,Donation button,header,,"Support us, Donate now",PayPal,top,,EUR,,
4,"'https://archive.ph/NDvuk',",https://buymeacoffee.com/archive.today,"donations, buy me a coffee",https://buymeacoffee.com/archive.today,"PayPal, credit cards","€1, €25, €50, €100",One-time donations only,Support Archive.Today,Buy Me a Coffee button,footer,Buy Me a Coffee,"Support us, Donate now",Buy Me a Coffee,bottom,donations,EUR,No subscription plans available,No subscription plans available
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,'https://correctiv.org/faktencheck/2024/05/27...,https://correctiv.org/unterstuetzen/verschenke...,donations,https://correctiv.org/unterstuetzen/?utm_sourc...,"SEPA direct debit, credit card, PayPal","50 €, 100 €, 200 €, 1000 €",Monthly or yearly donations available.,Give a donation to CORRECTIV,Donation button available on the website,Footer and dedicated support page,,"Support independent journalism, Donate now","PayPal, SEPA direct debit, credit card",Top of the donation form,"one-time donations, recurring donations",EUR,Monthly and yearly payment options available f...,Monthly and yearly payment options available f...
308,'https://correctiv.org/faktencheck/2024/05/27...,https://correctiv.org/unterstuetzen/unternehme...,donations,https://correctiv.org/unterstuetzen/?utm_sourc...,bank transfer,not specified,Information on one-time vs recurring donation ...,Your corporate donation flows directly into ou...,Donation buttons and links are provided throug...,The donation section is located in the main na...,,"Support us, Donate now",Bank for Social Economy,Buttons are located in the main content area a...,donations,EUR,not specified,not specified
312,'https://sso.globalnoticias.pt/realms/GMG_PRD...,https://forms.gle/mA3Ne9wBan9BQ2Wm9,,,,,NaN,NaN,NaN,NaN,,,,NaN,,,,
313,'https:

In [68]:
dd2 = dd1.drop(columns = ['Donation_Links'])

,Source_Url,Monetization_Url,Monetization_Method,Donation_Methods,Suggested_Donation_Amounts,Recurring_Donations_Option,Text_Related_To_Donations,Buttons_Or_Widgets,Placement_Of_Donation_Section,ThirdParty_Platforms,Call_To_Action_Messages,PaymentPlatform_Integration,Payment_Button_Widget_Location,Accepted_Payment_Methods,Pricing_Models,Currency_Support,Subscription_Plans
0,'https://vivementle9juin.fr/projet/europe-qui-...,https://adhesions.rassemblementnational.fr/,"donations, subscriptions","PayPal, credit cards","250 €, 100 €, 50 €, 30 €, 10 €",Donations can be one-time or recurring based o...,Support the RN!,Download the PDF form for donations and member...,Sidebar and footer of the page.,PayPal,"Join us!, Donate now!, Support us!",PayPal,Top of the donation section.,subscriptions,€,Prestige Membership: 250 € (85 € after tax red...,Prestige Membership: 250 € (85 € after tax red...
1,'https://triglavmedia.si/novice/okolje-ekolog...,https://triglavmedia.si/podprite-svobodo-govora,"donations, subscriptions","PayPal, credit cards, cryptocurrency","1€/month, 12€/year",Monthly or yearly support membership available.,Support freedom of speech with a donation or m...,Donate button on the Ko-fi platform.,Footer and dedicated donation page.,Ko-fi,"Support us, Donate now","PayPal, Ko-fi",Bottom of the page and on the donation page.,subscriptions,"EUR, USD",1€/month or 12€/year for membership,1€/month or 12€/year for membership
2,'https://lafranceinsoumise.fr/europeennes-202...,https://actionpopulaire.fr/contributions/,"donations, subscriptions",credit cards,,Recurring donations are available.,Become a supporter of La France insoumise — Po...,Donation button available on the page.,Footer of the page.,,"Support us, Donate now",Stripe,Bottom of the page.,subscriptions,EUR,"Monthly subscription for €5, Annual subscripti...","Monthly subscription for €5, Annual subscripti..."
3,'https://lafranceinsoumise.fr/europeennes-202...,https://actionpopulaire.fr/dons/,donations,PayPal,,One-time donations only,Make a donation,Donation button,header,,"Support us, Donate now",PayPal,top,,EUR,,
4,"'https://archive.ph/NDvuk',",https://buymeacoffee.com/archive.today,"donations, buy me a coffee","PayPal, credit cards","€1, €25, €50, €100",One-time donations only,Support Archive.Today,Buy Me a Coffee button,footer,Buy Me a Coffee,"Support us, Donate now",Buy Me a Coffee,bottom,donations,EUR,No subscription plans available,No subscription plans available
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,'https://correctiv.org/faktencheck/2024/05/27...,https://correctiv.org/unterstuetzen/verschenke...,donations,"SEPA direct debit, credit card, PayPal","50 €, 100 €, 200 €, 1000 €",Monthly or yearly donations available.,Give a donation to CORRECTIV,Donation button available on the website,Footer and dedicated support page,,"Support independent journalism, Donate now","PayPal, SEPA direct debit, credit card",Top of the donation form,"one-time donations, recurring donations",EUR,Monthly and yearly payment options available f...,Monthly and yearly payment options available f...
308,'https://correctiv.org/faktencheck/2024/05/27...,https://correctiv.org/unterstuetzen/unternehme...,donations,bank transfer,not specified,Information on one-time vs recurring donation ...,Your corporate donation flows directly into ou...,Donation buttons and links are provided throug...,The donation section is located in the main na...,,"Support us, Donate now",Bank for Social Economy,Buttons are located in the main content area a...,donations,EUR,not specified,not specified
312,'https://sso.globalnoticias.pt/realms/GMG_PRD...,https://forms.gle/mA3Ne9wBan9BQ2Wm9,,,,NaN,NaN,NaN,NaN,,,,NaN,,,,
313,'https://sso.globalnoticias.pt/realms/GMG_PRD...,https://forms.gle/DAK6B1tj8abq71gK9,,,,NaN,NaN,NaN,NaN,,,,NaN,,,,
